# 🍽️ UPES Hostel Food Waste Analysis
### Fundamentals of Data Science – Mini Project

**Dataset:** UPES Hostel-Wise Food Waste (6 hostels, 4 meals, ~46 days)

---
## 📋 Project in one line
We study **how much food is wasted in UPES hostels** and **what causes the waste**, using basic Python (Pandas, Matplotlib, Seaborn) and simple statistics.

## 🎯 Questions we answer
1. Which **hostel** wastes the most food?
2. Which **meal** (breakfast / lunch / snacks / dinner) wastes the most?
3. Do **weekends, exam periods or special events** change the waste?
4. Does **cooking extra food** (more leftovers) lead to more waste?
5. Can we **predict** waste (Linear Regression), **simplify** the data (PCA) and **classify** waste levels (LDA)?

## 🗺️ Steps we follow
`Load data → Clean data → Explore (EDA) → Visualise → Hypothesis tests → Linear Regression → PCA → LDA → Conclusion`

> 🧮 In Steps 7–9 we build Linear Regression, PCA and LDA **from their mathematical formulas using NumPy** (not by calling a ready-made function), so every step can be explained.

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd                 # tables / dataframes
import numpy as np                  # numbers
import matplotlib.pyplot as plt     # plotting
import seaborn as sns               # nicer plots
from scipy import stats             # statistical tests

sns.set_theme(style='whitegrid')
print('Libraries loaded ✅')

## 📂 Step 2: Load the Data
We read the Excel file into a Pandas **DataFrame** (a table).

> 💡 If you get a *file not found* error, put the Excel file in the same folder as this notebook and change `DATA_PATH` to just the file name.

In [ ]:
import os

DATA_PATH = '../data/raw/UPES_Hostel_Wise_Food_Waste_Synthetic_Dataset.xlsx'
if not os.path.exists(DATA_PATH):                      # fallback: same folder as notebook
    DATA_PATH = 'UPES_Hostel_Wise_Food_Waste_Synthetic_Dataset.xlsx'

df_raw = pd.read_excel(DATA_PATH)
print('Rows, Columns =', df_raw.shape)
df_raw.head()

### First look at the data

In [ ]:
df_raw.info()

**Important columns**

| Column | Meaning |
|---|---|
| `Hostel`, `Meal`, `Menu_Category` | Where, which meal, what food |
| `Students_Expected` | Students the kitchen expected |
| `Meals_Prepared` | Meals cooked |
| `Leftover_Meals` | Meals left over |
| **`Food_Waste_kg`** | **Food wasted in kg (our main variable)** |
| `Weekend`, `Exam_Period`, `Special_Event` | Yes / No flags |
| `Temperature_C` | Temperature of the day |

## 🧹 Step 3: Data Cleaning
Raw data is rarely perfect. We do 4 simple things:
1. Remove **empty / junk rows** (missing values)
2. Convert `Date` to a real date type
3. Convert **Yes/No → 1/0** so we can calculate with them
4. Remove **duplicate** rows

In [ ]:
df = df_raw.copy()

# 1. Missing values ----------------------------------------------------------
print('Missing values per column (before cleaning):')
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Convert Date (anything that is not a valid date becomes NaT = missing)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Drop every row that still has a missing value (blank rows, junk rows)
before = len(df)
df = df.dropna()
print(f'\nRows removed because of missing values: {before - len(df)}')

In [ ]:
# 3. Yes/No -> 1/0
for col in ['Weekend', 'Exam_Period', 'Special_Event', 'High_Waste_Event']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# 4. Duplicates
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print('Clean data shape:', df.shape)
df.head()

### New column (Feature creation)
We create **one** useful column: how many *extra* meals the kitchen cooked compared to the students expected.

`Extra_Meals = Meals_Prepared − Students_Expected`

In [ ]:
df['Extra_Meals'] = df['Meals_Prepared'] - df['Students_Expected']
df[['Students_Expected', 'Meals_Prepared', 'Extra_Meals', 'Food_Waste_kg']].head()

## 🔎 Step 4: Exploratory Data Analysis (EDA)
First the **summary statistics** (mean, std, min, max…) of the main numbers.

In [ ]:
df[['Food_Waste_kg', 'Meals_Prepared', 'Leftover_Meals', 'Extra_Meals', 'Temperature_C']].describe().round(2)

### Average waste by Hostel and by Meal (`groupby`)

In [ ]:
hostel_mean = df.groupby('Hostel')['Food_Waste_kg'].mean().sort_values(ascending=False).round(2)
meal_mean   = df.groupby('Meal')['Food_Waste_kg'].mean().sort_values(ascending=False).round(2)

print('Average waste (kg) by Hostel:')
print(hostel_mean)
print('\nAverage waste (kg) by Meal:')
print(meal_mean)

### Average waste: Weekend / Exam period / Special event (0 = No, 1 = Yes)

In [ ]:
for col in ['Weekend', 'Exam_Period', 'Special_Event']:
    print(col)
    print(df.groupby(col)['Food_Waste_kg'].mean().round(2).to_string())
    print()

## 📊 Step 5: Visualisation
Charts make patterns easy to see. We use only **7 simple charts**.

### Chart 1 – How is food waste distributed? (Histogram)

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Food_Waste_kg'], bins=25, kde=True, color='steelblue')
plt.axvline(df['Food_Waste_kg'].mean(), color='red', linestyle='--',
            label=f"Mean = {df['Food_Waste_kg'].mean():.2f} kg")
plt.title('Distribution of Food Waste')
plt.xlabel('Food Waste (kg)')
plt.legend()
plt.show()

**How to read it:** Most records waste a small amount; the red line shows the average.

### Chart 2 – Waste by Hostel (Bar chart)

In [ ]:
hostel_mean.plot(kind='bar', color='teal', edgecolor='black', figsize=(8, 4))
plt.title('Average Food Waste by Hostel')
plt.ylabel('Average Food Waste (kg)')
plt.xticks(rotation=0)
plt.show()

**How to read it:** The tallest bar is the hostel that wastes the most.

### Chart 3 – Waste by Meal (Bar chart)

In [ ]:
meal_mean.plot(kind='bar', color='orange', edgecolor='black', figsize=(8, 4))
plt.title('Average Food Waste by Meal')
plt.ylabel('Average Food Waste (kg)')
plt.xticks(rotation=0)
plt.show()

### Chart 4 – Hostel × Meal (Heatmap)

In [ ]:
pivot = df.pivot_table(values='Food_Waste_kg', index='Hostel', columns='Meal', aggfunc='mean')

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Average Food Waste (kg): Hostel × Meal')
plt.show()

**How to read it:** Darker colour = more waste. It shows which *hostel + meal* combination is worst.

### Chart 5 – Do leftovers mean more waste? (Scatter plot)

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(x='Leftover_Meals', y='Food_Waste_kg', data=df,
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'red'})
plt.title('Leftover Meals vs Food Waste')
plt.show()

**How to read it:** The points go up from left to right → more leftover meals = more food waste.

### Chart 6 – Correlation Heatmap

In [ ]:
cols = ['Food_Waste_kg', 'Meals_Prepared', 'Leftover_Meals', 'Extra_Meals', 'Temperature_C']
plt.figure(figsize=(7, 5))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()

**How to read it:** Values go from −1 to +1.
Close to **+1** → move together · close to **0** → no relation · close to **−1** → move in opposite directions.

### Chart 7 – Waste over time (Line chart)

In [ ]:
daily = df.groupby('Date')['Food_Waste_kg'].sum()

plt.figure(figsize=(11, 4))
daily.plot(color='purple')
plt.title('Total Food Waste per Day (all hostels)')
plt.xlabel('Date')
plt.ylabel('Total Waste (kg)')
plt.show()

## 🧪 Step 6: Hypothesis Testing
Charts *suggest* a difference. A statistical test tells us if the difference is **real** or just **random chance**.

**Simple idea:**
- **H₀ (null hypothesis):** there is *no* difference.
- **p-value:** the probability of seeing such a difference by chance.
- **Rule:** if **p < 0.05** → reject H₀ → the difference is **real (significant)**.

### Test 1 – T-test: does waste differ between two groups? (Weekend, Exam, Special Event)

In [ ]:
def ttest_for(col):
    yes = df[df[col] == 1]['Food_Waste_kg']
    no  = df[df[col] == 0]['Food_Waste_kg']
    t, p = stats.ttest_ind(yes, no, equal_var=False)
    print(f'{col}:  Yes mean = {yes.mean():.2f} kg | No mean = {no.mean():.2f} kg | p-value = {p:.4f}')
    print('   →', 'Significant difference ✅' if p < 0.05 else 'No significant difference ❌')
    print()

for c in ['Weekend', 'Exam_Period', 'Special_Event']:
    ttest_for(c)

### Test 2 – ANOVA: does waste differ across *many* groups? (Hostels, Meals)

In [ ]:
for col in ['Hostel', 'Meal']:
    groups = [g['Food_Waste_kg'].values for _, g in df.groupby(col)]
    f, p = stats.f_oneway(*groups)
    print(f'{col}:  F = {f:.2f} | p-value = {p:.4g}')
    print('   →', 'Significant difference ✅' if p < 0.05 else 'No significant difference ❌')
    print()

### Test 3 – Correlation: Leftover meals vs Food waste

In [ ]:
r, p = stats.pearsonr(df['Leftover_Meals'], df['Food_Waste_kg'])
print(f'Pearson r = {r:.3f} | p-value = {p:.4g}')
print('Strength:', 'Strong' if abs(r) > 0.6 else 'Moderate' if abs(r) > 0.3 else 'Weak',
      'positive' if r > 0 else 'negative', 'relationship')

**Which test when?**

| Situation | Test |
|---|---|
| Compare **2 groups** (e.g., weekend vs weekday) | T-test |
| Compare **3 or more groups** (e.g., 6 hostels) | ANOVA |
| Relationship between **2 numbers** | Correlation (Pearson r) |

## 🤖 Step 7: Linear Regression (from formulas)
**Goal:** predict `Food_Waste_kg` from other columns.

### Simple Linear Regression (1 input)
$$y = b_0 + b_1 x$$
$$b_1 = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sum (x_i-\bar{x})^2} \qquad b_0 = \bar{y} - b_1\bar{x}$$

**Model quality**
$$R^2 = 1-\frac{SS_{res}}{SS_{tot}} \qquad SS_{res}=\sum (y_i-\hat{y}_i)^2 \qquad SS_{tot}=\sum (y_i-\bar{y})^2$$
$R^2$ = share of the waste variation explained by the model (0 = useless, 1 = perfect).

In [ ]:
x = df['Leftover_Meals'].values.astype(float)
y = df['Food_Waste_kg'].values.astype(float)

x_mean, y_mean = x.mean(), y.mean()

# slope and intercept using the formulas
b1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
b0 = y_mean - b1 * x_mean

y_hat = b0 + b1 * x
ss_res = np.sum((y - y_hat) ** 2)
ss_tot = np.sum((y - y_mean) ** 2)
r2_simple = 1 - ss_res / ss_tot

print(f'Equation : Food_Waste_kg = {b0:.3f} + {b1:.3f} × Leftover_Meals')
print(f'R²       : {r2_simple:.3f}')
print(f'Meaning  : every 1 extra leftover meal adds about {b1:.2f} kg of waste.')

plt.figure(figsize=(7, 4.5))
plt.scatter(x, y, alpha=0.4, label='Data')
xs = np.linspace(x.min(), x.max(), 100)
plt.plot(xs, b0 + b1 * xs, color='red', linewidth=2, label='Regression line')
plt.xlabel('Leftover Meals'); plt.ylabel('Food Waste (kg)')
plt.title('Simple Linear Regression (our own formula)')
plt.legend(); plt.show()

### Multiple Linear Regression (many inputs) – Normal Equation
$$\hat{\beta} = (X^TX)^{-1}X^Ty$$
- $X$ = input columns (plus a column of 1s for the intercept)
- $\hat{\beta}$ = the coefficients (one number per input)

We split the data: **80% to train** the model and **20% to test** it on unseen data.

In [ ]:
features = ['Leftover_Meals', 'Extra_Meals', 'Temperature_C', 'Weekend', 'Exam_Period', 'Special_Event']

X_all = df[features].values.astype(float)
y_all = df['Food_Waste_kg'].values.astype(float)

# ---- 80/20 train-test split (random but repeatable) ----
rng = np.random.default_rng(42)
idx = rng.permutation(len(df))
cut = int(0.8 * len(df))
train_idx, test_idx = idx[:cut], idx[cut:]

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

# add a column of ones for the intercept
Xtr = np.column_stack([np.ones(len(X_train)), X_train])
Xte = np.column_stack([np.ones(len(X_test)),  X_test])

# Normal Equation:  beta = (XᵀX)⁻¹ Xᵀ y   (pinv = inverse that never crashes)
beta = np.linalg.pinv(Xtr.T @ Xtr) @ (Xtr.T @ y_train)

coef_table = pd.DataFrame({'Feature': ['Intercept'] + features, 'Coefficient': beta.round(4)})
coef_table

In [ ]:
# ---- Metrics from formulas ----
def r2_score_f(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

def rmse_f(y_true, y_pred):                       # Root Mean Squared Error
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae_f(y_true, y_pred):                        # Mean Absolute Error
    return np.mean(np.abs(y_true - y_pred))

pred_train = Xtr @ beta
pred_test  = Xte @ beta
r2_test    = r2_score_f(y_test, pred_test)

print(f'{"":8s}{"R²":>8s}{"RMSE":>10s}{"MAE":>10s}')
print(f'{"Train":8s}{r2_score_f(y_train, pred_train):8.3f}{rmse_f(y_train, pred_train):10.3f}{mae_f(y_train, pred_train):10.3f}')
print(f'{"Test":8s}{r2_test:8.3f}{rmse_f(y_test, pred_test):10.3f}{mae_f(y_test, pred_test):10.3f}')

plt.figure(figsize=(6, 5))
plt.scatter(y_test, pred_test, alpha=0.5)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Waste (kg)'); plt.ylabel('Predicted Waste (kg)')
plt.title('Test Data: Actual vs Predicted'); plt.legend(); plt.show()

**How to read it:** points close to the red dashed line = good predictions.
**Train vs Test:** if both R² values are similar, the model is **not overfitting**.
**RMSE / MAE:** the average prediction error in kg (smaller is better).

## 🔻 Step 8: PCA – Principal Component Analysis (from formulas)
**Goal:** many columns carry overlapping information (e.g., meals prepared, served, consumed). PCA **compresses them into a few new columns (principal components)** while keeping most of the information.

**Steps / formulas**
1. **Standardise:** $z = \dfrac{x-\bar{x}}{s}$  (so every column has equal importance)
2. **Covariance matrix:** $C = \dfrac{Z^TZ}{n-1}$
3. **Eigen-decomposition:** $C\,v = \lambda\,v$  → eigenvectors $v$ = principal components, eigenvalues $\lambda$ = variance carried by each
4. **Explained variance:** $\dfrac{\lambda_i}{\sum \lambda}$
5. **Project the data:** $\text{Scores} = Z \cdot V$

In [ ]:
pca_cols = ['Students_Expected', 'Meals_Prepared', 'Meals_Served', 'Leftover_Meals',
            'Consumed_Meals', 'Food_Waste_kg', 'Temperature_C']
Xp = df[pca_cols].values.astype(float)

# 1. standardise
Z = (Xp - Xp.mean(axis=0)) / Xp.std(axis=0, ddof=1)

# 2. covariance matrix
C = (Z.T @ Z) / (len(Z) - 1)

# 3. eigenvalues and eigenvectors (eigh is for symmetric matrices like C)
eig_vals, eig_vecs = np.linalg.eigh(C)

# sort from largest to smallest eigenvalue
order = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[order]
eig_vecs = eig_vecs[:, order]

# 4. explained variance
explained  = eig_vals / eig_vals.sum()
cumulative = np.cumsum(explained)

pca_table = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(eig_vals))],
    'Eigenvalue': eig_vals.round(3),
    'Explained %': (explained * 100).round(2),
    'Cumulative %': (cumulative * 100).round(2)
})
n_90 = int(np.argmax(cumulative >= 0.90)) + 1
print(f'👉 {n_90} components keep at least 90% of the information (from {len(pca_cols)} columns).')
pca_table

In [ ]:
# Scree plot
plt.figure(figsize=(8, 4))
plt.bar(range(1, len(explained) + 1), explained * 100, color='skyblue', edgecolor='black', label='Each PC')
plt.plot(range(1, len(explained) + 1), cumulative * 100, 'ro-', label='Cumulative')
plt.axhline(90, color='gray', linestyle='--')
plt.xlabel('Principal Component'); plt.ylabel('Explained Variance (%)')
plt.title('Scree Plot'); plt.legend(); plt.show()

**Loadings** show how much each original column contributes to a component (big number = important).

In [ ]:
loadings = pd.DataFrame(eig_vecs[:, :2], index=pca_cols, columns=['PC1', 'PC2']).round(3)
loadings

In [ ]:
# 5. project data onto the first two components
scores = Z @ eig_vecs[:, :2]

plt.figure(figsize=(8, 5.5))
sns.scatterplot(x=scores[:, 0], y=scores[:, 1], hue=df['Hostel'].values, palette='tab10', alpha=0.7)
plt.xlabel(f'PC1 ({explained[0]*100:.1f}%)'); plt.ylabel(f'PC2 ({explained[1]*100:.1f}%)')
plt.title('PCA: Data in 2 dimensions (coloured by Hostel)')
plt.legend(title='Hostel', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

**How to read it:** 
- **PC1** is the strongest "direction" in the data – usually the *overall size* of the meal (more students → more meals → more waste).
- Each dot is one record; dots that are close together are similar records.
- If a hostel forms its own cluster, that hostel behaves differently from the others.

## 🎯 Step 9: LDA – Linear Discriminant Analysis (from formulas)
**Goal:** PCA ignores labels. **LDA uses labels** and finds the directions that **separate the classes best**.

Our classes: we split `Food_Waste_kg` into **3 equal groups → Low / Medium / High waste**.
We then ask: *can we tell the waste level using only the other columns* (students expected, extra meals, leftovers, temperature)?

**Formulas**
1. Class mean $\mu_k$ and overall mean $\mu$
2. **Within-class scatter:** $S_W = \sum_k \sum_{x \in k} (x-\mu_k)(x-\mu_k)^T$  (how spread out each class is)
3. **Between-class scatter:** $S_B = \sum_k n_k (\mu_k-\mu)(\mu_k-\mu)^T$  (how far apart the classes are)
4. Solve the eigenproblem of $S_W^{-1}S_B$ → eigenvectors $w$ = **discriminant directions**
5. Keep the top (classes − 1) = 2 directions and project: $LD = X \cdot W$
6. **Classify** a new record to the class whose centre is nearest in LDA space.

In [ ]:
# --- create the classes (3 equal-sized groups) ---
df['Waste_Level'] = pd.qcut(df['Food_Waste_kg'], q=3, labels=['Low', 'Medium', 'High']).astype(str)
print(df['Waste_Level'].value_counts())

lda_cols = ['Students_Expected', 'Extra_Meals', 'Leftover_Meals', 'Temperature_C']
classes  = ['Low', 'Medium', 'High']

XL = df[lda_cols].values.astype(float)
yL = df['Waste_Level'].values

# same train / test split as before
XL_train, XL_test = XL[train_idx], XL[test_idx]
yL_train, yL_test = yL[train_idx], yL[test_idx]

In [ ]:
n_feat = XL_train.shape[1]
overall_mean = XL_train.mean(axis=0)

S_W = np.zeros((n_feat, n_feat))     # within-class scatter
S_B = np.zeros((n_feat, n_feat))     # between-class scatter

for c in classes:
    Xc  = XL_train[yL_train == c]
    mu_c = Xc.mean(axis=0)
    S_W += (Xc - mu_c).T @ (Xc - mu_c)
    diff = (mu_c - overall_mean).reshape(-1, 1)
    S_B += len(Xc) * (diff @ diff.T)

# eigenproblem of  S_W⁻¹ · S_B   (pinv = safe inverse)
lda_vals, lda_vecs = np.linalg.eig(np.linalg.pinv(S_W) @ S_B)
lda_vals, lda_vecs = lda_vals.real, lda_vecs.real

order = np.argsort(lda_vals)[::-1]
lda_vals, lda_vecs = lda_vals[order], lda_vecs[:, order]

W = lda_vecs[:, :2]                                  # top 2 discriminant directions
lda_share = lda_vals[:2] / lda_vals.sum()
print(f'LD1 separates the classes {lda_share[0]*100:.1f}% , LD2 {lda_share[1]*100:.1f}%')

# project data
LD_train = XL_train @ W
LD_test  = XL_test  @ W

In [ ]:
# --- classify: nearest class centre in LDA space ---
centroids = {c: LD_train[yL_train == c].mean(axis=0) for c in classes}

def lda_predict(LD):
    dist = np.column_stack([np.linalg.norm(LD - centroids[c], axis=1) for c in classes])
    return np.array(classes)[dist.argmin(axis=1)]

pred_L = lda_predict(LD_test)
lda_acc = np.mean(pred_L == yL_test)

print(f'✅ LDA test accuracy : {lda_acc*100:.1f}%   (random guessing would give ≈ 33%)')
print('\nConfusion matrix (rows = actual, columns = predicted):')
conf = pd.crosstab(pd.Series(yL_test, name='Actual'), pd.Series(pred_L, name='Predicted'))
conf.reindex(index=classes, columns=classes, fill_value=0)

In [ ]:
# plot all records in the 2 LDA dimensions
LD_all = XL @ W
plt.figure(figsize=(8, 5.5))
sns.scatterplot(x=LD_all[:, 0], y=LD_all[:, 1], hue=yL,
                hue_order=classes, palette={'Low': 'green', 'Medium': 'orange', 'High': 'red'}, alpha=0.6)
for c in classes:
    plt.scatter(*centroids[c], marker='X', s=250, edgecolor='black', color='white')
plt.xlabel('LD1'); plt.ylabel('LD2')
plt.title('LDA: Low / Medium / High Waste in 2 dimensions  (X = class centre)')
plt.legend(title='Waste Level'); plt.show()

**How to read it:** if the colours form separate groups, the classes are easy to tell apart from the input columns.
**PCA vs LDA:** PCA looks for *maximum variance* (no labels) · LDA looks for *maximum class separation* (uses labels).

### ✔️ Check: do our formulas match the ready-made library? (optional)
We compare our NumPy results with `scikit-learn`. Matching results prove our formulas are correct.

In [ ]:
try:
    from sklearn.linear_model import LinearRegression
    from sklearn.decomposition import PCA
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

    lr = LinearRegression().fit(X_train, y_train)
    print('Linear Regression – max coefficient difference :', np.abs(lr.coef_ - beta[1:]).max().round(8))

    pca_lib = PCA().fit(Z)
    print('PCA – max explained-variance difference        :', np.abs(pca_lib.explained_variance_ratio_ - explained).max().round(8))

    lda_lib = LinearDiscriminantAnalysis().fit(XL_train, yL_train)
    print(f'LDA – library accuracy {lda_lib.score(XL_test, yL_test)*100:.1f}%  vs  ours {lda_acc*100:.1f}%  (small difference is normal)')
except ImportError:
    print('scikit-learn is not installed – skipping this optional check.')

## ✅ Step 10: Key Findings & Recommendations

In [ ]:
print('=' * 60)
print('KEY FINDINGS (from our data)')
print('=' * 60)
print(f'1. Highest-waste hostel : {hostel_mean.index[0]}  ({hostel_mean.iloc[0]} kg on average)')
print(f'2. Lowest-waste hostel  : {hostel_mean.index[-1]}  ({hostel_mean.iloc[-1]} kg on average)')
print(f'3. Highest-waste meal   : {meal_mean.index[0]}  ({meal_mean.iloc[0]} kg on average)')
print(f'4. Leftover meals vs waste correlation : r = {r:.2f}')
print(f'5. Linear Regression : R² on unseen test data = {r2_test:.2f}')
print(f'6. PCA : {n_90} components keep 90% of the information of {len(pca_cols)} columns')
print(f'7. LDA : waste level (Low/Medium/High) predicted with {lda_acc*100:.0f}% test accuracy')
print()
print('RECOMMENDATIONS')
print('-' * 60)
print('• Cook according to the number of students expected – avoid extra meals.')
print(f'• Give extra attention to {hostel_mean.index[0]} (portion control, daily tracking).')
print(f'• Review the {meal_mean.index[0]} menu quantity.')
print('• Plan food quantity carefully for special events and exam days.')
print('• Use the regression equation to estimate expected waste and set daily targets.')

## 🏁 Conclusion
- Food waste is **not the same everywhere** – it depends on the **hostel**, the **meal** and special events (checked with statistical tests).
- **Leftover meals** and food waste are strongly related, so **cooking only what is needed** is the best way to reduce waste.
- **Linear Regression** can predict the waste in kg, **PCA** shows that many columns carry the same information, and **LDA** can classify a record into Low / Medium / High waste.
- All three models were built **from their formulas**, which shows we understand how they work – not only how to call them.

**Future scope:** try Logistic Regression / Decision Trees, add more variables (menu popularity, attendance forecast) and build a daily "how much to cook" tool.

---
## 🎤 Presentation Guide (for viva)

**Suggested flow (≈ 10–12 min):** Problem → Data → Cleaning → 3–4 key charts → Hypothesis tests → Linear Regression → PCA → LDA → Findings → Conclusion.

### Questions your professor may ask – with easy answers
| Question | Simple answer |
|---|---|
| What is your dataset? | Hostel-wise food waste of UPES: 6 hostels, 4 meals, ~46 days, with waste in kg. |
| How did you clean the data? | Removed empty/junk rows, converted dates, changed Yes/No to 1/0, removed duplicates. |
| What is EDA? | Exploring the data with summary statistics and charts to understand patterns before modelling. |
| What is a p-value? | Probability that the difference happened by chance. If p < 0.05 we call it significant. |
| T-test vs ANOVA? | T-test compares 2 groups; ANOVA compares 3 or more groups. |
| Correlation vs causation? | Correlation only shows two things move together; it does not prove one causes the other. |
| **What is Linear Regression?** | A straight-line model $y = b_0 + b_1x$ that predicts a number (waste in kg) from inputs. |
| **How are the coefficients found?** | By the Normal Equation $\hat\beta=(X^TX)^{-1}X^Ty$, which minimises the squared error. |
| **What is R²?** | The share of variation in waste explained by the model (0 to 1). |
| **RMSE vs MAE?** | Both are average prediction errors in kg; RMSE punishes big mistakes more. |
| **Why train/test split?** | To check the model on data it has never seen, so we know it is not just memorising (overfitting). |
| **What is PCA?** | A method that compresses many related columns into a few new uncorrelated ones (principal components). |
| **Why standardise before PCA?** | So columns with big numbers (e.g., meals) do not dominate columns with small numbers (e.g., waste in kg). |
| **What are eigenvalues / eigenvectors here?** | Eigenvectors = directions of the components; eigenvalues = how much variance each direction holds. |
| **What does the scree plot show?** | How much information each component keeps, helping us choose how many to use. |
| **What is LDA?** | A method that uses class labels to find directions that best separate the classes (Low/Medium/High waste). |
| **PCA vs LDA?** | PCA is unsupervised (max variance); LDA is supervised (max class separation). |
| **What are S_W and S_B?** | S_W = spread inside each class (want small); S_B = distance between class centres (want large). |
| What is your main finding? | Waste depends on hostel and meal, and more leftovers mean more waste → cook only what is needed. |